# 10 — Magnitude error vs magnitude: σ(m) curves and m5 fitting

## Purpose

This notebook studies the **photometric error budget** by plotting `σ(m)` vs `m`
for both `scienceFlux` and `psfFlux`, per band (u, g, r, i, z, y), and fits the
LSST noise model (Ivezić et al. 2019, Eq. 5) to extract:

- **m5**: the per-band 5σ limiting magnitude
- **σ_sys**: the systematic noise floor

The fit model used is the high-SNR approximation:

```
σ(m) = sqrt( (0.04 * 10^(0.4*(m - m5)))^2 + σ_sys^2 )
```

## Data sources

Four data layers are compared per Gaia category:

1. **diaObject catalogue** (`diaobj_catalogue_gaia_stable.csv` or `diaobj_catalogue.csv`)  
   — per-band mean fluxes and errors aggregated over all visits
2. **Single-visit src** (`data_FINK_BLOCK_LC_03/{cat}_src_src_joined_butler.parquet`)  
   — individual detections (markers: filled circles)
3. **Single-visit fp** (`data_FINK_BLOCK_LC_03/{cat}_fp_fp_joined_butler.parquet`)  
   — forced photometry (markers: +)

## Layout: 4 rows × 7 columns per Gaia category

- **Row 1**: scienceFlux — diaObject catalogue (per-band mean)
- **Row 2**: psfFlux — diaObject catalogue (per-band mean)
- **Row 3**: scienceFlux — individual src (●) and fp (+)
- **Row 4**: psfFlux — individual src (●) and fp (+)

Columns 1–6: one per band (u g r i z y). Column 7: all bands overlaid.

Each subplot shows the Ivezić (2019) σ(m) fit with fitted m5 and σ_sys.

---

- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab / IN2P3 / CNRS (Université Paris-Saclay)
- **Follows:** notebooks 01, 03, 07, 09
- **Creation date:** 2026-04-14
- **Last update:** 2026-04-14
- **Subject:** Fink/LSST photometric depth — m5 fitting per band and Gaia category


## 1. Imports & configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from scipy.optimize import curve_fit

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")

In [ ]:
!ls data_FINK_BLOCK_LC_03/

In [ ]:
# ── Notebook tag & paths ──────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_10"

# diaObject catalogues from notebook 01
DIR_DATA_01 = "data_FINK_BLOCK_LC_01"
FILE_DIAOBJ_GAIA = os.path.join(DIR_DATA_01, "diaobj_catalogue_gaia_stable.csv")
FILE_DIAOBJ_ALL = os.path.join(DIR_DATA_01, "diaobj_catalogue.csv")

# Per-object src/fp light curves from notebook 03
DIR_DATA_03 = "data_FINK_BLOCK_LC_03"

# Figure output directory
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)

# ── Gaia categories to iterate over ──────────────────────────────────────────
GAIA_CATEGORIES = [
    "gaia_star_stable_hq",
    "gaia_nophotgstar_stable_unknown_parallax",
    "gaia_star_variable",
]

# ── Photometric bands ─────────────────────────────────────────────────────────
BANDS = list("ugrizy")
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}

# ── Plot limits ────────────────────────────────────────────────────────────────
MAG_MIN = 15.0
MAG_MAX = 28.0
ERR_MIN = 0.0
ERR_MAX = 0.3

# ── AB zero-point in nJy ──────────────────────────────────────────────────────
F0_NJY = 3631.0e9  # 3631 Jy = 3.631e9 nJy

# ── Ivezic 2019 fit initial parameters ────────────────────────────────────────
# m5 initial guess per band (approximate LSST single-visit depths)
M5_INIT = {"u": 23.4, "g": 24.5, "r": 24.0, "i": 23.6, "z": 22.9, "y": 22.1}
SIGMA_SYS_INIT = 0.005

# ── Matplotlib global settings ────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 8,
    }
)

print(f"Figure directory : {os.path.abspath(DIR_FIGS)}")
print(f"Categories       : {GAIA_CATEGORIES}")

## 2. Helper functions

### 2.1 Flux → magnitude conversion (AB system)

All fluxes from Fink are in **nJy**.  The AB magnitude and its propagated error are:

```
m      = -2.5 * log10(F / F0)       with F0 = 3631 Jy = 3631.0e9 nJy
σ(m)   = (2.5 / ln 10) * (σ_F / F)  ≈ 1.0857 * (σ_F / F)
```

Only observations with `F > 0` are kept to avoid log domain errors.

In [ ]:
def flux_to_mag(flux_njy, flux_err_njy=None, f0=F0_NJY):
    """
    Convert flux in nJy to AB magnitude and magnitude error.

    Parameters
    ----------
    flux_njy : array-like
    flux_err_njy : array-like
    f0 : float
        AB zero-point in nJy (default 3.631e9)

    Returns
    -------
    mag : ndarray  (NaN where flux <= 0)
    mag_err : ndarray  (NaN where flux <= 0)
    """
    flux = np.asarray(flux_njy, dtype=float)
    ferr = np.asarray(flux_err_njy, dtype=float)
    mag = np.full_like(flux, np.nan)
    mag_err = np.full_like(flux, np.nan)
    ok = flux > 0
    mag[ok] = -2.5 * np.log10(flux[ok] / f0)
    mag_err[ok] = (2.5 / np.log(10)) * np.abs(ferr[ok] / flux[ok])
    return mag, mag_err


print("flux_to_mag() defined.")

### 2.2 Ivezić (2019) σ(m) noise model and fitter

Equation 5 from Ivezić et al. 2019 (arXiv:0805.2366), high-SNR approximation:

$$
\sigma(m) = \sqrt{\left(0.04 \cdot 10^{0.4(m - m_5)}\right)^2 + \sigma_{\rm sys}^2}
$$

Free parameters: `m5` (5σ limiting magnitude) and `σ_sys` (systematic noise floor).

In [ ]:
def sigma_model(m, m5, sigma_sys):
    """
    Ivezić et al. 2019 photometric noise model (high-SNR approximation, Eq. 5).

    σ(m) = sqrt( (0.04 * 10^(0.4*(m - m5)))^2 + sigma_sys^2 )

    Parameters
    ----------
    m : array-like  — magnitude values
    m5 : float      — 5-sigma limiting magnitude
    sigma_sys : float — systematic noise floor

    Returns
    -------
    sigma : ndarray
    """
    x = 10.0 ** (0.4 * (np.asarray(m, dtype=float) - m5))
    return np.sqrt((0.04 * x) ** 2 + sigma_sys**2)


def fit_m5_sigma(mag, mag_err, band="?", m5_init=None, sigma_sys_init=0.01, mag_min=MAG_MIN, mag_max=MAG_MAX):
    """
    Fit the Ivezić σ(m) model to (mag, mag_err) data.

    Parameters
    ----------
    mag, mag_err : array-like
    band : str      — photometric band label (used for m5 initial guess)
    m5_init : float — override m5 initial guess (None = use M5_INIT[band])
    sigma_sys_init : float — initial sigma_sys
    mag_min, mag_max : float — restrict fit to this magnitude range

    Returns
    -------
    popt : (m5, sigma_sys) or None if fit fails
    perr : 1-σ uncertainties on popt, or None
    """
    mag = np.asarray(mag, dtype=float)
    mag_err = np.asarray(mag_err, dtype=float)
    finite = np.isfinite(mag) & np.isfinite(mag_err) & (mag_err > 0)
    in_range = (mag >= mag_min) & (mag <= mag_max)
    mask = finite & in_range
    if mask.sum() < 5:
        return None, None
    m_fit = mag[mask]
    s_fit = mag_err[mask]
    # Clip extreme outliers (99th percentile on errors)
    p99 = np.nanpercentile(s_fit, 99)
    good = s_fit < p99
    if good.sum() < 5:
        return None, None
    m_fit = m_fit[good]
    s_fit = s_fit[good]
    # Initial guess
    if m5_init is None:
        m5_init = M5_INIT.get(band, 23.5)
    p0 = [m5_init, sigma_sys_init]
    try:
        popt, pcov = curve_fit(
            sigma_model,
            m_fit,
            s_fit,
            p0=p0,
            bounds=([16.0, 0.0], [27.0, 0.5]),
            maxfev=5000,
        )
        perr = np.sqrt(np.diag(pcov))
        return popt, perr
    except RuntimeError:
        return None, None


print("sigma_model() and fit_m5_sigma() defined.")

### 2.3 Subplot drawing helper

In [ ]:
def plot_sigma_vs_mag(
    ax, mag, mag_err, band, color, label=None, marker="o", ms=2, alpha=0.4, do_fit=True, m5_init=None
):
    """
    Scatter plot σ(m) vs m on ax and overlay the fitted model.

    Returns fitted (m5, sigma_sys) tuple or (None, None) if fit fails.
    """
    mag = np.asarray(mag, dtype=float)
    mag_err = np.asarray(mag_err, dtype=float)
    finite = np.isfinite(mag) & np.isfinite(mag_err) & (mag_err > 0)
    mag = mag[finite]
    mag_err = mag_err[finite]

    ax.scatter(
        mag,
        mag_err,
        s=ms**2,
        color=color,
        alpha=alpha,
        marker=marker,
        linewidths=0,
        label=label,
        rasterized=True,
    )

    m5_fit = sigma_sys_fit = None
    if do_fit and len(mag) >= 5:
        popt, perr = fit_m5_sigma(mag, mag_err, band=band, m5_init=m5_init)
        if popt is not None:
            m5_fit, sigma_sys_fit = popt
            m_arr = np.linspace(MAG_MIN, MAG_MAX, 200)
            ax.plot(
                m_arr,
                sigma_model(m_arr, m5_fit, sigma_sys_fit),
                color=color,
                lw=1.5,
                ls="--",
                label=f"m5={m5_fit:.2f}, σ_sys={sigma_sys_fit:.4f}",
            )

    ax.set_xlim(MAG_MIN, MAG_MAX)
    ax.set_ylim(ERR_MIN, ERR_MAX)
    return m5_fit, sigma_sys_fit


print("plot_sigma_vs_mag() defined.")

## 3. Load data

### 3.1 diaObject catalogues

In [ ]:
# Load the best available diaObject catalogue
if os.path.exists(FILE_DIAOBJ_GAIA):
    df_diaobj = pd.read_csv(FILE_DIAOBJ_GAIA)
    diaobj_label = "diaobj_catalogue_gaia_stable"
    print(f"Loaded: {FILE_DIAOBJ_GAIA}  →  {df_diaobj.shape}")
elif os.path.exists(FILE_DIAOBJ_ALL):
    df_diaobj = pd.read_csv(FILE_DIAOBJ_ALL)
    diaobj_label = "diaobj_catalogue"
    print(f"Loaded: {FILE_DIAOBJ_ALL}  →  {df_diaobj.shape}")
else:
    raise FileNotFoundError(
        f"No diaObject catalogue found in {DIR_DATA_01}.\n"
        "Run notebook 01_fink_block_flatlightcurves.ipynb first."
    )

print("Columns (first 30):", df_diaobj.columns[:30].tolist())
print(
    "Groups in catalogue:",
    df_diaobj["group"].value_counts().to_dict() if "group" in df_diaobj.columns else "n/a",
)

In [ ]:
# ── Build per-band (mean_flux, mean_flux_err) from diaObject catalogue ─────────
# The catalogue contains columns like:
#   r:{b}_scienceFluxMean, r:{b}_scienceFluxMeanErr
#   r:{b}_psfFluxMean,     r:{b}_psfFluxMeanErr
# where b in {u,g,r,i,z,y}

DIAOBJ_SCI_FLUX_COL = "r:{b}_scienceFluxMean"
DIAOBJ_SCI_ERR_COL = "r:{b}_scienceFluxMeanErr"
DIAOBJ_PSF_FLUX_COL = "r:{b}_psfFluxMean"
DIAOBJ_PSF_ERR_COL = "r:{b}_psfFluxMeanErr"

# Verify a band exists
test_col = DIAOBJ_SCI_FLUX_COL.format(b="r")
if test_col in df_diaobj.columns:
    print(f"OK: column '{test_col}' found in diaObject catalogue.")
else:
    # Fallback: look for alternative naming
    print(f"WARNING: '{test_col}' NOT found.  Inspecting available columns ...")
    sci_cols = [c for c in df_diaobj.columns if "scienceFlux" in c or "psfFlux" in c]
    print("  sci/psf flux columns found:", sci_cols[:20])

In [ ]:
AB_FLUX_ZERO = 3631e9  # nJy at AB zero-point


def flux_to_mag_v0(flux_nJy, flux_err_nJy=None):
    flux = np.asarray(flux_nJy, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        mag = np.where(flux > 0, -2.5 * np.log10(flux / AB_FLUX_ZERO), np.nan)
    mag_err = None
    if flux_err_nJy is not None:
        err = np.asarray(flux_err_nJy, dtype=float)
        with np.errstate(invalid="ignore", divide="ignore"):
            mag_err = np.where(flux > 0, 2.5 / np.log(10) * np.abs(err / flux), np.nan)
    return mag, mag_err


def plot_diaobjmag_histograms_by_band(df):
    # Définir les bandes et les colonnes correspondantes
    bands = ["u", "g", "r", "i", "z", "y"]
    science_columns = [f"r:{band}_scienceFluxMean" for band in bands]
    psf_columns = [f"r:{band}_psfFluxMean" for band in bands]

    scienceerr_columns = [f"r:{band}_scienceFluxMeanErr" for band in bands]
    psferr_columns = [f"r:{band}_psfFluxMeanErr" for band in bands]

    # Créer la figure et les sous-graphes (1 ligne, 2 colonnes)
    fig, axes = plt.subplots(1, 2, figsize=(8, 3), sharex=True)

    # Définir une couleur par bande
    BAND_COLORS = {"u": "blue", "g": "green", "r": "red", "i": "purple", "z": "orange", "y": "brown"}

    # Pour chaque bande, calculer les magnitudes et tracer les histogrammes
    for band, sci_col, psf_col in zip(bands, science_columns, psf_columns):
        bcolor = BAND_COLORS.get(band, "gray")

        # ScienceFluxMean (à gauche)
        flux_science = df[sci_col].dropna().values
        mag_science, _ = flux_to_mag_v0(flux_science)
        axes[0].hist(
            mag_science[~np.isnan(mag_science)],
            bins=30,
            color=bcolor,
            alpha=1,
            lw=3,
            histtype="step",
            label=f"{band} (science)",
        )

        # psfFluxMean (à droite)
        flux_psf = df[psf_col].dropna().values
        mag_psf, _ = flux_to_mag_v0(flux_psf)
        axes[1].hist(
            mag_psf[~np.isnan(mag_psf)],
            bins=30,
            color=bcolor,
            alpha=1,
            lw=3,
            histtype="step",
            label=f"{band} (psf)",
        )

    # Configuration des axes et légendes
    for ax in axes:
        ax.set_xlabel("Magnitude")
        ax.set_ylabel("Count")
        ax.legend()

    axes[0].set_title("scienceFluxMean (Gaia stars)")
    axes[1].set_title("psfFluxMean (Gaia stars)")

    plt.tight_layout()

    plt.show()

In [ ]:
def plot_diaobjmagerrmag_scatter_by_band(df):
    # Définir les bandes et les colonnes correspondantes
    bands = ["u", "g", "r", "i", "z", "y"]
    science_columns = [f"r:{band}_scienceFluxMean" for band in bands]
    psf_columns = [f"r:{band}_psfFluxMean" for band in bands]

    scienceerr_columns = [f"r:{band}_scienceFluxMeanErr" for band in bands]
    psferr_columns = [f"r:{band}_psfFluxMeanErr" for band in bands]

    # Créer la figure et les sous-graphes (1 ligne, 2 colonnes)
    fig, axes = plt.subplots(1, 2, figsize=(8, 3), sharex=True, sharey=True)

    # Définir une couleur par bande
    BAND_COLORS = {"u": "blue", "g": "green", "r": "red", "i": "purple", "z": "orange", "y": "brown"}

    # Pour chaque bande, calculer les magnitudes et tracer les histogrammes
    for band, sci_col, scierr_col, psf_col, psferr_col in zip(
        bands, science_columns, scienceerr_columns, psf_columns, psferr_columns
    ):
        bcolor = BAND_COLORS.get(band, "gray")

        # ScienceFluxMean (à gauche)

        flux_science = df[sci_col]
        flux_scienceerr = df[scierr_col]
        mask = (~np.isnan(flux_science)) & (~np.isnan(flux_scienceerr))
        mag_science, magerr_science = flux_to_mag_v0(flux_science[mask], flux_scienceerr[mask])
        axes[0].scatter(
            mag_science, magerr_science, color=bcolor, marker="+", lw=1, label=f"{band} (science)"
        )

        # psfFluxMean (à droite)
        flux_psf = df[psf_col]
        flux_psferr = df[psferr_col]
        mask = (~np.isnan(flux_psf)) & (~np.isnan(flux_psferr))

        mag_psf, magerr_psf = flux_to_mag_v0(flux_psf[mask], flux_psferr[mask])
        axes[1].scatter(mag_psf, magerr_psf, color=bcolor, marker="+", lw=1, label=f"{band} (psf)")

    # Configuration des axes et légendes
    for ax in axes:
        ax.set_xlabel("Magnitude")
        ax.set_ylabel("Magnitude Error")
        ax.set_ylim(0.0, 0.5)
        ax.legend()

    axes[0].set_title("scienceFluxMean (Gaia stars)")
    axes[1].set_title("psfFluxMean (Gaia stars)")

    plt.tight_layout()

    plt.show()

In [ ]:
plot_diaobjmag_histograms_by_band(df_diaobj)

In [ ]:
plot_diaobjmagerrmag_scatter_by_band(df_diaobj)

### 3.2 Per-category src and fp parquet files

In [ ]:
# Discover available parquet files per category
# Pattern: {cat}_src_joined_butler.parquet  /  {cat}_fp_joined_butler.parquet
# Fall back to _consdb if _butler not present.


def load_parquet_for_cat(cat, kind):
    """
    Load src or fp joined parquet for a given Gaia category.

    Parameters
    ----------
    cat  : str  e.g. 'gaia_star_stable_hq'
    kind : str  'src' or 'fp'

    Returns
    -------
    df : DataFrame or None
    """
    for suffix in ("butler", "consdb"):
        fname = f"{cat}_{kind}_joined_{suffix}.parquet"
        fpath = os.path.join(DIR_DATA_03, fname)
        if os.path.exists(fpath):
            df = pd.read_parquet(fpath)
            print(f"  Loaded {fpath}  →  {df.shape}")
            return df
    print(f"  WARNING: no parquet found for {cat}/{kind} in {DIR_DATA_03}")
    return None


# Pre-load all data into dicts keyed by category
data_src = {}
data_fp = {}

print("Loading src parquets ...")
for cat in GAIA_CATEGORIES:
    data_src[cat] = load_parquet_for_cat(cat, "src")

print("\nLoading fp parquets ...")
for cat in GAIA_CATEGORIES:
    data_fp[cat] = load_parquet_for_cat(cat, "fp")

In [ ]:
# Auto-detect scienceFlux and psfFlux column names from the first available src DataFrame

COL_BAND = "r:band"
COL_PSF = None
COL_PSFERR = None
COL_SCI = None
COL_SCIERR = None

PSF_CANDIDATES = ["r:psfFlux", "psfFlux"]
PSFERR_CANDIDATES = ["r:psfFluxErr", "psfFluxErr"]
SCI_CANDIDATES = ["r:scienceFlux", "r:psfScienceFlux", "scienceFlux", "psfScienceFlux"]
SCIERR_CANDIDATES = ["r:scienceFluxErr", "r:scienceSigma", "scienceFluxErr", "scienceSigma"]

# Use the first available DataFrame to discover column names
ref_df = next((v for v in data_src.values() if v is not None), None)
if ref_df is not None:
    for c in PSF_CANDIDATES:
        if c in ref_df.columns:
            COL_PSF = c
            break
    for c in PSFERR_CANDIDATES:
        if c in ref_df.columns:
            COL_PSFERR = c
            break
    for c in SCI_CANDIDATES:
        if c in ref_df.columns:
            COL_SCI = c
            break
    for c in SCIERR_CANDIDATES:
        if c in ref_df.columns:
            COL_SCIERR = c
            break

print(f"Band column      : {COL_BAND}")
print(f"psfFlux          : {COL_PSF}")
print(f"psfFluxErr       : {COL_PSFERR}")
print(f"scienceFlux      : {COL_SCI}")
print(f"scienceFluxErr   : {COL_SCIERR}")

## 4. Magnitude extraction helpers

In [ ]:
def get_diaobj_mag_for_band_flux(df, band, flux_col_tpl, err_col_tpl):
    """
    Extract (mag, mag_err) from the diaObject catalogue for one band.

    Parameters
    ----------
    df : DataFrame   — the diaObject catalogue
    band : str       — e.g. 'r'
    flux_col_tpl : str  — template like 'r:{b}_scienceFluxMean'
    err_col_tpl  : str  — template like 'r:{b}_scienceFluxMeanErr'

    Returns
    -------
    mag, mag_err : ndarray (NaN-filtered)
    """
    fc = flux_col_tpl.format(b=band)
    ec = err_col_tpl.format(b=band)
    if fc not in df.columns or ec not in df.columns:
        return np.array([]), np.array([])
    flux = df[fc].values.astype(float)
    ferr = df[ec].values.astype(float)
    mag, mag_err = flux_to_mag(flux, np.abs(ferr))
    finite = np.isfinite(mag) & np.isfinite(mag_err) & (mag_err > 0)
    return mag[finite], mag_err[finite]


def get_src_mag_for_band(df, band, col_flux, col_err):
    """
    Extract (mag, mag_err) from a src/fp DataFrame for one band.

    Parameters
    ----------
    df : DataFrame
    band : str
    col_flux, col_err : str  — column names for flux and error

    Returns
    -------
    mag, mag_err : ndarray
    """
    if df is None or col_flux is None or col_err is None:
        return np.array([]), np.array([])
    if COL_BAND not in df.columns:
        return np.array([]), np.array([])
    mask = df[COL_BAND] == band
    sub = df[mask]
    if sub.empty:
        return np.array([]), np.array([])
    if col_flux not in sub.columns or col_err not in sub.columns:
        return np.array([]), np.array([])
    flux = sub[col_flux].values.astype(float)
    ferr = sub[col_err].values.astype(float)
    mag, mag_err = flux_to_mag(flux, np.abs(ferr))
    finite = np.isfinite(mag) & np.isfinite(mag_err) & (mag_err > 0)
    return mag[finite], mag_err[finite]


print("Extraction helpers defined.")

## 5. Main plotting loop

For each Gaia category, produce a **4 × 7** figure:

| Row | Flux type | Data source |
|-----|-----------|-------------|
| 1   | scienceFlux | diaObject catalogue (per-band mean) |
| 2   | psfFlux     | diaObject catalogue (per-band mean) |
| 3   | scienceFlux | individual visits: src ● and fp + |
| 4   | psfFlux     | individual visits: src ● and fp + |

Columns 1–6: individual bands (u g r i z y). Column 7: all bands overlaid.

In [ ]:
def make_figure_for_category(cat):
    """
    Produce the 4×7 figure for a given Gaia category.

    Returns
    -------
    fig : matplotlib Figure
    fit_results : dict  keyed by (row, band) → (m5, sigma_sys)
    """
    # ── Data selection ────────────────────────────────────────────────────────
    # diaObject rows for this category
    if "group" in df_diaobj.columns:
        df_cat = df_diaobj[df_diaobj["group"] == cat].copy()
    else:
        df_cat = df_diaobj.copy()

    df_src_cat = data_src.get(cat, None)
    df_fp_cat = data_fp.get(cat, None)

    n_diaobj = len(df_cat)
    n_src = len(df_src_cat) if df_src_cat is not None else 0
    n_fp = len(df_fp_cat) if df_fp_cat is not None else 0
    print(f"\n{'=' * 60}")
    print(f"Category: {cat}")
    print(f"  diaObjects : {n_diaobj}")
    print(f"  src rows   : {n_src}")
    print(f"  fp rows    : {n_fp}")

    # ── Figure layout ─────────────────────────────────────────────────────────
    NROWS, NCOLS = 4, 7
    fig, axes = plt.subplots(
        NROWS,
        NCOLS,
        figsize=(22, 12),
        sharex=True,
        sharey=True,
        constrained_layout=True,
    )

    # Row labels
    row_labels = [
        "scienceFlux\n(diaObj)",
        "psfFlux\n(diaObj)",
        "scienceFlux\n(src ● / fp +)",
        "psfFlux\n(src ● / fp +)",
    ]
    for r, lbl in enumerate(row_labels):
        axes[r, 0].set_ylabel(lbl + "\n" + r"$\sigma_m$ [mag]", fontsize=7)

    for c, band in enumerate(BANDS):
        axes[0, c].set_title(f"{band}-band", fontsize=9, fontweight="bold", color=BAND_COLORS[band])
    axes[0, 6].set_title("all bands", fontsize=9, fontweight="bold", color="0.3")

    for c in range(NCOLS):
        axes[NROWS - 1, c].set_xlabel("mag [AB]", fontsize=7)

    fig.suptitle(
        f"σ(m) vs m — category: {cat}\nIvezić fit: σ = √[(0.04 · 10^{{0.4(m–m5)}})² + σ_sys²]",
        fontsize=11,
    )

    fit_results = {}  # (row_idx, band) → (m5, sigma_sys)

    # ── Loop over rows ────────────────────────────────────────────────────────
    for row_idx in range(NROWS):
        is_diaobj = row_idx < 2  # rows 0,1 = diaObject; rows 2,3 = single visit
        is_sci = row_idx in (0, 2)  # rows 0,2 = scienceFlux; rows 1,3 = psfFlux

        # Select diaObj column templates
        if is_sci:
            diaobj_flux_tpl = DIAOBJ_SCI_FLUX_COL
            diaobj_err_tpl = DIAOBJ_SCI_ERR_COL
            src_flux_col = COL_SCI
            src_err_col = COL_SCIERR
        else:
            diaobj_flux_tpl = DIAOBJ_PSF_FLUX_COL
            diaobj_err_tpl = DIAOBJ_PSF_ERR_COL
            src_flux_col = COL_PSF
            src_err_col = COL_PSFERR

        # Column 7 (index 6): all-bands overlay
        ax_all = axes[row_idx, 6]

        for col_idx, band in enumerate(BANDS):
            ax = axes[row_idx, col_idx]
            color = BAND_COLORS[band]

            if is_diaobj:
                # diaObject: one point per object (per-band mean flux)
                mag, mag_err = get_diaobj_mag_for_band_flux(df_cat, band, diaobj_flux_tpl, diaobj_err_tpl)
                if len(mag) > 0:
                    m5_fit, ssys = plot_sigma_vs_mag(
                        ax, mag, mag_err, band, color, marker="o", ms=4, alpha=0.6, do_fit=True
                    )
                    plot_sigma_vs_mag(
                        ax_all, mag, mag_err, band, color, marker="o", ms=3, alpha=0.3, do_fit=False
                    )
                    fit_results[(row_idx, band)] = (m5_fit, ssys)
                    if m5_fit is not None:
                        ax.set_title(f"{band}: m5={m5_fit:.2f}", fontsize=7, color=color)
                ax.text(
                    0.97,
                    0.95,
                    f"N={len(mag)}",
                    transform=ax.transAxes,
                    ha="right",
                    va="top",
                    fontsize=6,
                    color="0.5",
                )

            else:
                # Individual visits: src (filled circle) + fp (plus marker)
                mag_src, err_src = get_src_mag_for_band(df_src_cat, band, src_flux_col, src_err_col)
                mag_fp, err_fp = get_src_mag_for_band(df_fp_cat, band, src_flux_col, src_err_col)

                all_mag = (
                    np.concatenate([mag_src, mag_fp]) if len(mag_src) > 0 or len(mag_fp) > 0 else np.array([])
                )
                all_err = (
                    np.concatenate([err_src, err_fp]) if len(err_src) > 0 or len(err_fp) > 0 else np.array([])
                )

                m5_fit = ssys = None
                if len(mag_src) > 0:
                    plot_sigma_vs_mag(
                        ax,
                        mag_src,
                        err_src,
                        band,
                        color,
                        marker="o",
                        ms=2,
                        alpha=0.3,
                        do_fit=False,
                        label="src",
                    )
                    plot_sigma_vs_mag(
                        ax_all, mag_src, err_src, band, color, marker="o", ms=1.5, alpha=0.15, do_fit=False
                    )
                if len(mag_fp) > 0:
                    plot_sigma_vs_mag(
                        ax, mag_fp, err_fp, band, color, marker="+", ms=2, alpha=0.3, do_fit=False, label="fp"
                    )
                    plot_sigma_vs_mag(
                        ax_all, mag_fp, err_fp, band, color, marker="+", ms=1.5, alpha=0.15, do_fit=False
                    )
                # Fit on combined src+fp data
                if len(all_mag) >= 5:
                    popt, perr = fit_m5_sigma(all_mag, all_err, band=band)
                    if popt is not None:
                        m5_fit, ssys = popt
                        m_arr = np.linspace(MAG_MIN, MAG_MAX, 200)
                        ax.plot(
                            m_arr,
                            sigma_model(m_arr, m5_fit, ssys),
                            color=color,
                            lw=1.5,
                            ls="--",
                            label=f"m5={m5_fit:.2f}",
                        )
                        ax.set_title(f"{band}: m5={m5_fit:.2f}", fontsize=7, color=color)
                fit_results[(row_idx, band)] = (m5_fit, ssys)
                ax.text(
                    0.97,
                    0.95,
                    f"src={len(mag_src)} fp={len(mag_fp)}",
                    transform=ax.transAxes,
                    ha="right",
                    va="top",
                    fontsize=5.5,
                    color="0.5",
                )

            # Legend on individual-band panels for single-visit rows
            if not is_diaobj and col_idx == 0:
                hdl_src = mlines.Line2D([], [], color=color, marker="o", ls="None", ms=4, label="src")
                hdl_fp = mlines.Line2D([], [], color=color, marker="+", ls="None", ms=5, label="fp")
                ax.legend(handles=[hdl_src, hdl_fp], fontsize=6, loc="upper left")

        # All-band subplot: fit on all data combined
        if is_diaobj:
            # Collect all-band data
            all_mag_ab, all_err_ab = [], []
            for band in BANDS:
                m, e = get_diaobj_mag_for_band_flux(df_cat, band, diaobj_flux_tpl, diaobj_err_tpl)
                all_mag_ab.extend(m.tolist())
                all_err_ab.extend(e.tolist())
            all_mag_ab = np.array(all_mag_ab)
            all_err_ab = np.array(all_err_ab)
            if len(all_mag_ab) >= 5:
                popt, _ = fit_m5_sigma(all_mag_ab, all_err_ab, band="r")
                if popt is not None:
                    m_arr = np.linspace(MAG_MIN, MAG_MAX, 200)
                    ax_all.plot(
                        m_arr,
                        sigma_model(m_arr, *popt),
                        color="0.2",
                        lw=2,
                        ls="--",
                        label=f"all: m5={popt[0]:.2f}",
                    )
                    ax_all.legend(fontsize=6)

    return fig, fit_results


def savefig(name):
    """Save current figure as PDF and PNG in DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight", dpi=130)
    print(f"  → saved {name}.{{pdf,png}}")


print("make_figure_for_category() defined.")

## 6. Run: loop over all Gaia categories

In [ ]:
all_fit_results = {}

for cat in GAIA_CATEGORIES:
    fig, fits = make_figure_for_category(cat)
    all_fit_results[cat] = fits
    fig_name = f"sigma_vs_mag_{cat}"
    savefig(fig_name)
    plt.show()

print("\nAll figures produced.")

## 7. Summary table: fitted m5 and σ_sys per category and band

In [ ]:
# Row descriptions matching the 4-row layout
ROW_LABELS = {
    0: "scienceFlux / diaObj",
    1: "psfFlux     / diaObj",
    2: "scienceFlux / src+fp",
    3: "psfFlux     / src+fp",
}

records = []
for cat, fits in all_fit_results.items():
    for (row_idx, band), (m5, ssys) in fits.items():
        records.append(
            {
                "category": cat,
                "data_layer": ROW_LABELS.get(row_idx, f"row{row_idx}"),
                "band": band,
                "m5_fit": round(m5, 3) if m5 is not None else np.nan,
                "sigma_sys": round(ssys, 5) if ssys is not None else np.nan,
            }
        )

df_summary = pd.DataFrame(records)
df_summary = df_summary.sort_values(["category", "data_layer", "band"])
print(df_summary.to_string(index=False))

# Save summary
summary_path = os.path.join(DIR_FIGS, "m5_fit_summary.csv")
df_summary.to_csv(summary_path, index=False)
print(f"\nSummary saved to: {summary_path}")

## 8. Comparison plot: m5 per band across Gaia categories

In [ ]:
# Compare diaObject-level m5 fits across the three categories
# Focus on rows 0 (scienceFlux) and 1 (psfFlux)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True, constrained_layout=True)
fig.suptitle("Fitted m5 per band — comparison across Gaia categories (diaObject level)", fontsize=11)

MARKERS_CAT = {cat: mk for cat, mk in zip(GAIA_CATEGORIES, ["o", "s", "^"])}
x_ticks = np.arange(len(BANDS))

for ax_idx, row_idx in enumerate([0, 1]):
    ax = axes[ax_idx]
    ax.set_title(["scienceFlux", "psfFlux"][ax_idx] + " (diaObj)", fontsize=10)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(BANDS)
    ax.set_xlabel("Band")
    ax.set_ylabel("Fitted $m_5$ [AB mag]")
    ax.set_ylim(18, 27)

    for cat in GAIA_CATEGORIES:
        fits = all_fit_results.get(cat, {})
        m5_vals = [fits.get((row_idx, b), (None, None))[0] for b in BANDS]
        m5_vals = [v if v is not None else np.nan for v in m5_vals]
        ax.plot(x_ticks, m5_vals, marker=MARKERS_CAT[cat], lw=1.5, label=cat, ms=7)

    # Reference LSST single-visit m5 from Ivezić 2019 / SMTN-002
    m5_ref = [M5_INIT[b] for b in BANDS]
    ax.plot(
        x_ticks,
        m5_ref,
        marker="*",
        lw=1.5,
        ls=":",
        color="0.4",
        ms=10,
        label="LSST reference m5 (single visit)",
    )

    ax.legend(fontsize=7, loc="lower right")
    ax.grid(True, alpha=0.4)

savefig("m5_comparison_per_band")
plt.show()

## Notes

- The `σ_sys` floor reflects the **template noise** contribution for `psfFlux`
  (difference-image flux), and the **calibration repeatability** for `scienceFlux`.
- For `scienceFlux`, values of `m5` close to the nominal LSST single-visit depth
  (u≈23.4, g≈24.5, r≈24.0, i≈23.6, z≈22.9, y≈22.1) are expected for well-behaved
  conditions.  Deviations indicate non-standard conditions or template contamination.
- The `gaia_nophotgstar_stable` category has unknown parallax and may include
  extragalactic contaminants; its σ(m) curve will appear broader than the
  pure-stellar categories.
- Adjust `MAG_MIN`, `MAG_MAX`, `ERR_MIN`, `ERR_MAX` in the configuration cell
  to zoom in on specific regions of the σ(m) diagram.
